# Instruments :Stocks, Futures , FX , Crypto ,Options

You can apply the same quant ideas to wildly different things: a share of Apple, a barrel of oil delivered in December, the euro against the dollar, Bitcoin, or the right to buy a stock at a fixed price. Each of these is an instrument, and each behaves differently in terms of cost, leverage, settlement, and risk. Choosing the right instrument for your strategy and your account is a decision that quietly determines whether you succeed.This lesson gives you a working map of the main instrument classes.

By the end of this lesson you will be able to 
- Describe what each major instrument class is and how it settles
- Explain how leverage and costs differ across instruments
- Match an instrument to a typical quant use case
- Weigh the pros and cons of each for an individual trader
- Read a side-by-side comparison and pick sensibly for your situation


## 1. Stock(Equities)

A **stock** is a fractional ownership stake in a company. If you buy one share of a firm with a billion shares, you own a billionth of it.

- **Settlement**. In the US, stock trades settle one business day after the trade (referred to as T+1). You become the legal owner shortly after trading.
- **Leverage**. Limited for individuals — a standard margin account allows roughly 2x intraday and 2x overnight under Reg T. Cash accounts use no leverage.
- **Costs**. Commissions are often zero at retail brokers, but you still pay the bid-ask spread, and possibly borrow fees if you short. Liquid large-caps have tiny spreads.
- **Quant use**. Cross-sectional strategies — ranking hundreds or thousands of stocks and going long the best, short the worst (momentum, value, mean reversion). Equities are the classic playground for factor investing.
- **Pros / cons**. Pro: huge universe, deep data, easy to understand. Con: limited leverage, short selling can be restricted or expensive, and you face overnight gap risk on earnings and news.

A subtle point that matters for backtesting: a stock's price is not a clean signal of return, because it gets distorted by corporate actions. When a company does a 2-for-1 split, the price halves overnight even though nothing of value changed; when it pays a dividend, the price drops by roughly the dividend amount. If you compute returns from raw closing prices, these create fake "crashes" that never happened. This is why you almost always work with the **adjusted close**, which rewrites history to remove split and dividend artifacts, leaving a series whose percentage changes are true investor returns. We flagged this in the environment lesson and it recurs throughout the course.

## 2. Futures
A **future** is a standardised contract to buy or sell an asset (an index, oil, gold, bonds, even Bitcoin) at a set price on a set future date. You don't own the underlying; you hold a contract whose value moves with it.

- **Settlement**. Many are cash-settled (you exchange the profit or loss in cash); some are physically settled. Contracts expire, so you must roll to the next month to keep a position.

- **Leverage**. High and built-in. You post a margin deposit that is a small fraction of the contract's notional value, so a small price move is large relative to your capital.

- **Costs**. Low commissions and very tight spreads on liquid contracts; the main hidden cost is the roll and the spread.

- **Quant use**. Trend following and macro strategies across many asset classes with one uniform instrument type. CTAs (managed futures funds) live here.

- **Pros / cons**. Pro: deep liquidity, near-24-hour trading, capital efficiency, easy to go short. Con: leverage cuts both ways and can wipe small accounts fast; contract specs and rolls add complexity.

The roll deserves a concrete picture because it confuses newcomers and quietly corrupts naive backtests. Each futures contract has an expiry; to hold a position past it, you close the expiring contract and open the next month's — that's "rolling." The two contracts usually trade at different prices, so the roll itself moves your effective entry point. Worse, a price chart stitched from raw contract prices shows a jump at each roll that isn't a real gain or loss. Data providers solve this with a **continuous contract** that adjusts for the roll gaps, much like the adjusted close does for stock dividends. If you backtest futures on un-rolled raw prices, you'll measure phantom profits at every expiry — a classic and expensive mistake.


## 3. Foreign Exchange(FX)

**FX** is trading one currency against another — for example EUR/USD, the number of US dollars per euro. It is the largest market in the world by volume.

- **Settlement**. Spot FX nominally settles in two days, but retail positions are rolled automatically and you rarely take delivery.
- **Leverage**. Often very high at retail brokers (sometimes 30x to 100x+ depending on jurisdiction), which is a major reason beginners blow up.
- **Costs**. Mostly the spread, plus a small overnight financing charge (swap) for holding positions, reflecting the interest-rate difference between the two currencies.
- **Quant use**. Carry strategies (earning the rate differential), trend following, and mean reversion on highly liquid pairs. Macro and rate-driven models.
- **Pros / cons**. Pro: enormous liquidity, 24-hour weekday trading, cheap to access. Con: extreme leverage tempts oversizing; retail FX is full of poorly regulated brokers; pairs are driven by hard-to-predict macro forces.

One conceptual hurdle in FX is that every quote is a ratio of two things, so there is no absolute "up." When EUR/USD rises, that can mean the euro strengthened, the dollar weakened, or both — and a trade in EUR/USD is simultaneously a long euro position and a short dollar position. This is why FX is inseparable from interest rates: holding a currency means effectively lending in it and borrowing the other, so you earn or pay the rate differential overnight (the "swap"). The carry strategy is built directly on this mechanic — you hold the higher-yielding currency to collect the differential — which also means FX returns are entangled with central-bank policy in a way single stocks are not.



## 4. Cryptocurrency

**Crypto** assets like Bitcoin and Ethereum are digital tokens traded on dedicated exchanges, 24/7, with no central regulator and no closing bell.

- **Settlement**. Spot trades settle almost instantly on the exchange's ledger; you can withdraw to your own wallet. Derivatives (perpetual futures) dominate volume and never expire, using a funding rate to tether them to spot.
- **Leverage**. Often extremely high (10x to 100x on some venues), with fast automated liquidations.
- **Costs**. Taker/maker fees that are higher than equities, plus sometimes wide spreads on smaller coins and meaningful slippage.
- **Quant use**. Momentum and trend on majors, statistical arbitrage between pairs and venues, funding-rate harvesting. A common entry point for individuals because APIs are open and data is free.
- **Pros / cons**. Pro: 24/7, open APIs, easy to start, can short via perpetuals. Con: high volatility, exchange and custody risk (an exchange can fail), thin liquidity outside majors, regulatory uncertainty.


The dominant crypto instrument by volume is not spot at all but the **perpetual future** ("perp"), and it's worth understanding because it has no equivalent in traditional markets. A perp tracks the spot price but never expires, so to keep it tethered there's a **funding rate**: periodically (often every eight hours), longs and shorts pay each other a small amount depending on whether the perp is trading above or below spot. When the perp is above spot, longs pay shorts; when below, shorts pay longs. This creates an entire strategy family — "funding harvesting" — where you hold the paid side while hedging the price risk. It also means holding a leveraged perp position has an ongoing carry cost or credit you must include in any backtest, exactly analogous to the FX swap and the futures roll.


## 5. Options

An **option** is the right, but not the obligation, to buy (a call) or sell (a put) an underlying asset at a fixed strike price before or at expiration. You pay a premium for that right.

- **Settlement**. Equity options are typically physically settled into shares; index options are cash-settled. They expire on set dates.
- **Leverage**. Implicit and nonlinear — a small premium controls a large notional, but value depends on price, time, and volatility together.
- **Costs**. Spreads can be wide, commissions are per-contract, and time decay steadily erodes long option value.
- **Quant use**. Volatility trading (selling premium, dispersion), hedging tail risk, and constructing defined-risk payoffs. Requires understanding the "Greeks."
- **Pros / cons**. Pro: precise, asymmetric payoffs and powerful hedging. Con: genuinely complex; the most common way beginners lose money quickly. Treat as advanced.

What makes options fundamentally different from everything above is that their value depends on more than just the price of the underlying. A stock position makes or loses money in a straight line as the price moves. An option's value depends jointly on the price, on how much time is left (value bleeds away as expiration approaches — "theta"), and on how much volatility the market expects ("vega"). This is the trap that catches beginners: you can correctly predict that a stock will rise, buy a call, watch the stock rise modestly, and still lose money because time decay and falling volatility ate your premium faster than the price move added to it. Being right on direction is not enough; you must be right on direction, magnitude, and timing. That's why we treat options as advanced and devote a later module to the Greeks.


## 6. Comparision Table

|Instrument	|What it is	|Settlement|	Typical leverage	|Main cost	|Common quant use|	Beginner-friendly?|
|---|---|---|---|---|---|---|
|Stocks	|Company ownership|	T+1|	~2x	|Spread (low)|	Cross-sectional factors|	Yes|
|Futures	|Contract on an asset	|Cash/physical, expires	|High (built-in)	|Spread + roll	|Trend following, macro	|Moderate|
|FX	|Currency vs currency|	T+2 (rolled)|	Very high|	Spread + swap|	Carry, trend|	Moderate|
|Crypto	|Digital token|	Near-instant / perpetual|	Very high|	Fees + slippage|	Momentum, stat-arb, funding|	Yes (with care)|
|Options	|Right to buy/sell	|Physical/cash, expires	|High, nonlinear|	|Spread + decay|	Volatility, hedging|	No (advanced)|


## 7. Worked Example:Same idea, different instrument

Say you want to bet that gold will rise over the next month. You have three realistic routes:

1. **Buy a gold ETF (stock-like)**. Simple, no expiry, no leverage by default. $10,000 buys $10,000 of exposure. A 3% gold move makes you about $300, minus a tiny spread.
2. **Buy one gold future**. A single contract controls a large notional, but you only post margin of a few thousand dollars. The same 3% move could be several thousand dollars of profit or loss on a few thousand of margin — wonderful when right, account-ending when wrong.
3. **Buy a gold call option**. You pay a small premium for the upside. If gold rises sharply you can multiply your premium; if it drifts sideways, time decay can take the whole premium even though you were "right" about direction.

Same view, three completely different risk profiles. The instrument is part of the strategy, not an afterthought.

Let's put numbers on it to feel how different these routes really are. Assume gold is at $2,000/oz, you have $10,000, and gold rises exactly 3% over the month.

In [1]:
capital = 10000
gold_move = 0.3

# Route 1: ETF, no leverage
etf_pnl = capital * gold_move
etf_return = etf_pnl / capital


# Route 2: One future . Gold Furure = 100oz. so notional = 2000 *100
notional = 2000 * 100 # $200,000 controlled
margin = 11_000 # approx initial margin posted
fut_pnl = notional * gold_move
fut_return = fut_pnl / margin # return on the margin you actually posted


# Route 3: a call option costing 400$ that ends ~3x. if right ,worthless if not.
premium = 400
opt_pnl_if_right = premium * 2      # net gain (value ~3x premium, minus the premium paid)
opt_pnl_if_flat = -premium          # full loss if gold drifts sideways

print(f"ETF: P&L{etf_pnl:0.8f} return {etf_return:6.1%}")
print(f"Future : P&L{fut_pnl:0.8f} return {fut_return:6.1%}")
print(f"Option (right): P&L {opt_pnl_if_right:6.0f}")
print(f"Option (flat):  P&L {opt_pnl_if_flat:6.0f}")


ETF: P&L3000.00000000 return  30.0%
Future : P&L60000.00000000 return 545.5%
Option (right): P&L    800
Option (flat):  P&L   -400


The ETF earns a sober 3%. The single future earns $6,000 on $11,000 of posted margin — roughly a 55% return — but note that the same 3% move in the other direction loses $6,000, more than half your account, and a larger move could trigger a margin call. The option either multiplies your small premium or vaporizes it entirely depending on path and timing. One identical market view, three radically different outcomes: the instrument choice is a risk-sizing decision.

## Common Mistakes

- Reaching for maximum leverage because the broker allows it. Available leverage is a limit, not a target; high leverage is the leading cause of blown accounts.
- Ignoring carry and roll costs. FX swaps, futures rolls, and crypto funding quietly erode returns on held positions and must be in your backtest.
- Trading options before understanding them. Being right on direction and still losing money is the norm for beginners; the Greeks are not optional knowledge.
- Assuming all crypto is liquid. Majors are deep; smaller tokens can have spreads and slippage that destroy any edge.
- Forgetting expiry. Futures and options expire — a strategy that ignores rolls and expiration dates will break in live trading.